# How good is the search? Vectors alone vs vectors + metadata

Each ask is run twice over the posters: **vector search alone** (CLIP text vs poster embeddings) and **the same vector search restricted
by what Google's detectors found** (family-friendly ratings, nobody looks unhappy).

Two kinds of check, both printed below:

* **Constraint check** (from the stored detector values): how many results have an unhappy face or a rating of POSSIBLE or higher. The filtered
  search should have none: this is a guarantee, not a surprise.
* **Independent check**: the poster's *genres* in the public TMDB 5000 dataset (not produced by Google's detectors). "Animation" and "Family"
  are the true answer to *"an animated family movie"*, so the share of results that carry them says whether the ranking itself is good.

Needs the CLIP embeddings (`python embed.py`, optional requirements in `requirements-embeddings.txt`) and internet once, for the TMDB metadata.

In [ ]:
import ast
import csv
import io
import os
import urllib.request
import zipfile

import pandas as pd

import config
import vector_search as vs

client = config.get_client()
DATASET = None
K = 10
ASKS = [
    "an animated family movie",
    "a family-friendly animated movie where everyone looks happy",
    "a cheerful cartoon movie for kids",
]

## Ground truth: TMDB genres (public)

In [ ]:
URL = "https://www.kaggle.com/api/v1/datasets/download/tmdb/tmdb-movie-metadata"
CACHE = "tmdb_5000_movies.csv"
if not os.path.exists(CACHE):
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=120).read())) as z:
        open(CACHE, "wb").write(z.read("tmdb_5000_movies.csv"))
genres = {r["id"]: ([g["name"] for g in ast.literal_eval(r["genres"])], r["title"])
          for r in csv.DictReader(open(CACHE, encoding="utf-8"))}
len(genres)

## The comparison

In [ ]:
def stem(row):
    return row["gcs_object_name"].rsplit("/", 1)[-1].rsplit(".", 1)[0]


def score(rows):
    g = [genres.get(stem(r), ([], "?"))[0] for r in rows]
    return {
        "animation": sum("Animation" in x for x in g),
        "family": sum("Family" in x for x in g),
        "unhappy face": sum((r.get("face_count") or 0) > 0 and (r.get("min_face_joy") or 0) < 0.6 for r in rows),
        "rated POSSIBLE+": sum((r.get("adult_content_rank") or 1) > 2 or (r.get("violent_content_rank") or 1) > 2 for r in rows),
    }


variants = {"vector only": None, "vector + metadata": vs.family_criteria(max_rank=2, min_joy=0.6)}
table, titles = [], {}
for ask in ASKS:
    for name, criteria in variants.items():
        rows, _ = vs.search(client, ask, k=K, criteria=criteria, dataset=DATASET)
        table.append({"ask": ask, "search": name, "results": len(rows), **score(rows)})
        titles[(ask, name)] = ", ".join(genres.get(stem(r), ([], "?"))[1] for r in rows)
result = pd.DataFrame(table)
result

*animation* and *family* count how many of the results are tagged with that genre in TMDB (out of `results`). The filter removes every
unhappy-face and POSSIBLE+ result; whether it also improves the genre match depends on the ask, so all asks tried are shown, not a selection.

## What came back

In [ ]:
for (ask, name), t in titles.items():
    print(f"{name:18} | {ask}\n    {t}\n")